# Module 7 - Session 2: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To solidify the mathematical concepts of MDPs and the Bellman equation through conceptual and calculation-based problems.

## Setup

These exercises are theoretical. You only need a pen and paper or a text editor. The small Python cells below are included only to verify the arithmetic.


## Exercise 1: Conceptual Questions (30 minutes)

### Foundation

An MDP requires a state representation that contains all information needed to predict the next state and reward. If important information is missing, the Markov Property does not hold.

### Build

1. **The Markov Property**

For **Blackjack**, the state defined by **player hand total**, **dealer visible card**, and **usable ace** is usually treated as Markov in the simplified RL version of the game. In that standard formulation, the future depends only on the current hand situation, not on the full history of how the cards were dealt. However, in real Blackjack with a finite deck, this is not perfectly Markov because the cards already seen change the composition of the remaining deck. That means the past can still matter.

For **Poker**, defining a Markov state is much harder because there is significant **hidden information**. You do not know the opponents' private cards, and betting history carries important information about what they might be holding. A useful state would need not only the visible cards and pot information, but also some belief about hidden hands and opponent behavior. That makes the true Markov state much more difficult to represent.

2. **The Discount Factor (`gamma`)**

Agent A, with `gamma = 0.1`, is very short-sighted. It heavily discounts future rewards, so the `+100` exit reward loses value quickly if it is many steps away. Agent B, with `gamma = 0.99`, values future rewards much more strongly, so it is more willing to take a sequence of actions that leads to a better long-term result. Agent B is more likely to learn the **absolute shortest path**, because the delayed exit reward still matters a lot and the step penalties encourage reducing path length.

3. **Policies**

A **stochastic policy** can be better when unpredictability matters. In a game like **Rock-Paper-Scissors**, a deterministic policy would be easy for an opponent to exploit because they could learn your fixed pattern. A stochastic policy avoids that by randomizing actions. Stochastic policies are also useful in uncertain or partially observable settings, where spreading probability across several reasonable actions can be better than always committing to one move.

### Result

The key ideas are that state design determines whether an MDP is valid, `gamma` controls how far ahead the agent plans, and stochastic policies are often valuable when predictability is a weakness.


## Exercise 2: The Bellman Equation in a Grid World (60 minutes)

We have a simple `3 x 1` grid world:

`s1 (+10)   s2 (Start)   s3 (-10)`

- States: `S = {s1, s2, s3}`
- `s1` and `s3` are terminal states
- Actions: `A = {left, right}`
- Discount factor: `gamma = 0.9`
- Policy `pi`: in `s2`, go left with probability `0.5` and right with probability `0.5`


### Solution (Markdown Answer)

### Foundation

The Bellman Expectation Equation says that the value of a state under a policy equals the expected immediate reward plus the discounted value of the next state.

### Build

1. **Value of the terminal states**

Once the agent reaches a terminal state, the episode ends. That means there is **no future reward after that point**, so:

- `V_pi(s1) = 0`
- `V_pi(s3) = 0`

The `+10` and `-10` rewards are received **when entering** those states, not after already being in them.

2. **Value of the start state `V_pi(s2)`**

Using the Bellman Expectation Equation:

`V_pi(s2) = sum_a pi(a|s2) * [ R(s2, a) + gamma * V_pi(s') ]`

Under this policy:

- With probability `0.5`, the agent goes left to `s1` and gets reward `+10`
- With probability `0.5`, the agent goes right to `s3` and gets reward `-10`

So:

`V_pi(s2) = 0.5 * [10 + 0.9 * V_pi(s1)] + 0.5 * [-10 + 0.9 * V_pi(s3)]`

Substitute the terminal values:

`V_pi(s2) = 0.5 * [10 + 0.9 * 0] + 0.5 * [-10 + 0.9 * 0]`

`V_pi(s2) = 0.5 * 10 + 0.5 * (-10)`

`V_pi(s2) = 5 - 5 = 0`

Therefore:

- `V_pi(s2) = 0`

### Result

This makes sense because the policy gives equal probability to a `+10` outcome and a `-10` outcome, so the expected value balances out to zero.


In [1]:
gamma = 0.9
V_pi_s1 = 0
V_pi_s3 = 0

V_pi_s2 = 0.5 * (10 + gamma * V_pi_s1) + 0.5 * (-10 + gamma * V_pi_s3)

print(f"V_pi(s1) = {V_pi_s1}")
print(f"V_pi(s3) = {V_pi_s3}")
print(f"V_pi(s2) = {V_pi_s2}")


V_pi(s1) = 0
V_pi(s3) = 0
V_pi(s2) = 0.0


## Exercise 3: Challenge Problem - Policy Evaluation (30 minutes)

Now consider a new policy `pi'`:

- In `s2`, go left with probability `0.8`
- In `s2`, go right with probability `0.2`


### Analysis (Markdown Answer)

### Foundation

Policy evaluation means computing the value function for a fixed policy. Here, only the action probabilities have changed.

### Build

1. **Calculate `V_pi'(s2)`**

Using the same Bellman equation:

`V_pi'(s2) = 0.8 * [10 + 0.9 * V_pi'(s1)] + 0.2 * [-10 + 0.9 * V_pi'(s3)]`

Since `s1` and `s3` are still terminal:

`V_pi'(s1) = 0` and `V_pi'(s3) = 0`

So:

`V_pi'(s2) = 0.8 * 10 + 0.2 * (-10)`

`V_pi'(s2) = 8 - 2 = 6`

2. **Compare the two policies**

From Exercise 2:

- `V_pi(s2) = 0`
- `V_pi'(s2) = 6`

The new policy `pi'` is better because it chooses the action leading to the positive terminal reward much more often. That improves the expected return from the start state. This makes intuitive sense because `left` leads to `+10` and `right` leads to `-10`, so a better policy should prefer `left` more strongly.

### Result

Policy evaluation gives a direct numerical way to compare policies. In this grid world, increasing the probability of moving toward the good terminal state raises the value of the start state from `0` to `6`.


In [2]:
V_pi_prime_s1 = 0
V_pi_prime_s3 = 0

V_pi_prime_s2 = 0.8 * (10 + gamma * V_pi_prime_s1) + 0.2 * (-10 + gamma * V_pi_prime_s3)

print(f"V_pi(s2) = {V_pi_s2}")
print(f"V_pi'(s2) = {V_pi_prime_s2}")
print(f"Better policy: {'pi\'' if V_pi_prime_s2 > V_pi_s2 else 'pi'}")


V_pi(s2) = 0.0
V_pi'(s2) = 6.0
Better policy: pi'
